# Course Case Competition Workflow — Forecasting the S&P 500

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2027S_predictive_analytics_QM474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>QM47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2027S_predictive_analytics_QM474/blob/main/notebooks/nb18_competition_workflow_student.ipynb)

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises**. You are expected to complete both exercises before submitting your notebook.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Pull the S&P 500 yourself with an explicit **information cutoff**, so every forecast uses only what was known at its origin.
2. Express a forecast as **quantiles** of the index's cumulative log return at 1, 5, 10, 15 and 20 trading days — the competition's submission format.
3. Score quantile forecasts with the competition's **scaled pinball loss** and compare them with the **random-walk benchmark**.
4. Run a **weekly walk-forward backtest** in which every model faces the identical origins, and describe the **paired per-origin differences** honestly.
5. Check the **calibration** of your forecast bands before trusting them.
6. Produce a **reproducible, validated submission file** for the Kaggle practice round and for the live forecast.

---

## Setup

The setup cell loads the libraries and the competition's scoring rule. The function `score` below is the **same function** the Kaggle practice round and the live round use — copied from the competition's published scorer — so the numbers you compute in your backtest are on exactly the scale of the leaderboard. Nothing needs installing in Colab.

In [ ]:
# Setup — everything below ships with Google Colab (no installs needed)
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")
RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)
pd.set_option("display.precision", 3)

# ---- The competition's scoring rule (identical to the published scorer) ----
TAUS = np.array([0.10, 0.25, 0.50, 0.75, 0.90])
QCOLS = ["q10", "q25", "q50", "q75", "q90"]
HORIZONS = [1, 5, 10, 15, 20]          # trading days after the origin close
Z = stats.norm.ppf(TAUS)               # standard-normal quantiles


def make_id(origin, h):
    return f"{pd.Timestamp(origin):%Y-%m-%d}_h{int(h):02d}"


def score(solution, submission, row_id_column_name="id"):
    """Scaled pinball loss (lower is better). solution: id, y, h, scale, origin. submission: id + QCOLS."""
    rid = row_id_column_name
    sub = submission.set_index(rid).loc[solution[rid], QCOLS].to_numpy(dtype=float)
    if not np.isfinite(sub).all() or (np.diff(sub, axis=1) < 0).any():
        raise ValueError("Quantiles must be finite and non-decreasing (q10 <= q25 <= q50 <= q75 <= q90)")
    u = solution["y"].to_numpy(dtype=float)[:, None] - sub
    pinball = np.maximum(TAUS * u, (TAUS - 1.0) * u)
    row_loss = pinball.mean(axis=1) / (solution["scale"].to_numpy(dtype=float)
                                       * np.sqrt(solution["h"].to_numpy(dtype=float)))
    return float(pd.Series(row_loss).groupby(solution["origin"].astype(str).to_numpy()).mean().mean())


def describe_paired(gains, label):
    """Describe paired per-origin gains (positive = the challenger was better). No interval, no verdict."""
    g = np.asarray(gains, dtype=float)
    return pd.Series({"origins": len(g), "mean gain": round(g.mean(), 4), "SD of gains": round(g.std(ddof=1), 4),
                      "origins won": f"{int((g > 0).sum())} of {len(g)}",
                      "largest gain": round(g.max(), 4), "largest loss": round(g.min(), 4)}, name=label)


print("Setup complete!")
print(f"RANDOM_SEED = {RANDOM_SEED}")

**Reading the output:** `Setup complete!` means the libraries and the scoring rule loaded. You will call `score` on your own backtest in Section 5.

---

## 1. Why This Matters: A Range the Treasury Officer Can Plan Around

In nb16 the treasury officer of the state pension fund learned that no model read the market's direction better than the random walk, but that the **size** of the next month's move is forecastable. Now the request is precise:

> *"Every Thursday I decide how much cash to hold against a fall in our stock portfolio. I do not need you to call the market. Give me, for the next day and for each of the next four weeks, the level the S&P 500 return will stay above nine times out of ten, the level it will stay below nine times out of ten, and the middle — and show me your method would have done that honestly every week of the last two years."*

That request **is** the Course Case Competition. Your group (your final-project group, named `Group NN` on Kaggle) submits, for each horizon $h \in \{1, 5, 10, 15, 20\}$ trading days after the origin close, five quantiles — **q10, q25, q50, q75, q90** — of the cumulative log return in percent,

$$y(T, h) = 100 \cdot \ln\!\left(\frac{P_{T+h}}{P_T}\right).$$

The live origin is the close of **Thursday, March 25, 2027**; the NYSE is closed for Good Friday on March 26, when your forecast and code are due. The five horizons land on **Mon Mar 29, Fri Apr 2, Fri Apr 9, Fri Apr 16 and Fri Apr 23** — the next trading day and one to four weeks. Before that, a **practice round on Kaggle** scores you on weekly origins whose outcomes are already known. The full rules, deadlines and grading are in the competition instructions on Brightspace; this notebook is the workflow.

> **A question that often comes up here:** *"Why quantiles instead of a single predicted price?"* Because a single number is the part of the forecast nobody can get right. nb16 showed that the direction of the next month is close to a coin flip, so a point forecast is mostly luck. A set of quantiles describes the whole range, and the scoring rule rewards a range that is honest (the truth falls below q10 about 10% of the time) and tight (no wider than it needs to be). That is a skill you can build and test.

---

## 2. Pull the Data with an Explicit Information Cutoff

`DATA_END` is the most important line in your competition notebook. It is the last close your forecast is allowed to know. In this notebook it is fixed at **October 30, 2026** so your numbers match the text; for the live forecast your group sets it to **"2027-03-25"** and changes nothing else. The loader downloads daily closes from Yahoo Finance through `yfinance` (preinstalled in Colab) and refuses to return anything after the cutoff. The index values belong to S&P Dow Jones Indices: you download them yourself for your coursework, and you never upload them anywhere — not to Kaggle, not to Brightspace, not to GitHub.

In [ ]:
DATA_START = "2018-01-01"
DATA_END = "2026-10-30"      # <- for the live forecast, set to "2027-03-25" and re-run everything


def load_sp500(start=DATA_START, end=DATA_END, csv_path=None):
    """Daily S&P 500 closes from `start` through `end` inclusive (Yahoo Finance via yfinance).
    If Yahoo is unavailable, download a CSV yourself (columns Date, Close) and pass csv_path."""
    if csv_path is not None:
        raw = pd.read_csv(csv_path, parse_dates=["Date"]).set_index("Date")["Close"]
    else:
        import yfinance as yf
        end_exclusive = (pd.Timestamp(end) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        raw = yf.download("^GSPC", start=start, end=end_exclusive,
                          auto_adjust=False, progress=False)["Close"].squeeze()
    raw.index = pd.to_datetime(raw.index)
    if raw.index.tz is not None:
        raw.index = raw.index.tz_localize(None)
    close = raw.loc[start:end].dropna().astype(float)
    assert close.index.max() <= pd.Timestamp(end), "data after the information cutoff!"
    close.name = "close"
    return close


close = load_sp500()
ret = 100 * np.log(close).diff().dropna()      # daily log returns in PERCENT

print(f"Closes: {len(close):,} trading days, {close.index[0]:%Y-%m-%d} -> {close.index[-1]:%Y-%m-%d}")
print(f"Last close before the cutoff: {close.iloc[-1]:,.2f} on {close.index[-1]:%A %Y-%m-%d}")
print(f"Daily log return (%): mean {ret.mean():.3f}, SD {ret.std():.3f}")
print(f"SD of the last 252 daily returns (the competition's scale s_T): {ret.iloc[-252:].std(ddof=1):.3f}%")

**Reading the output:** The demo cutoff returns **2,220 trading days, from 2 January 2018 to Friday 30 October 2026** (last close **7,580.07**). Daily log returns average **0.047%** with an SD of **1.205%** over the whole window, but the SD of the **last 252** days — the competition's scale $s_T$ at this cutoff — is only **0.808%**: the past year was calmer than the 2018–2026 average. That difference is why the scale is computed from the recent year at each origin rather than from all history.

---

## 3. The Score: Pinball Loss, Scaled to a Typical Move

The **pinball loss** of a quantile forecast $q_\tau$ when the truth is $y$ is

$$L_\tau(y, q_\tau) = \max\big(\tau\,(y - q_\tau),\; (\tau - 1)\,(y - q_\tau)\big).$$

For the median ($\tau = 0.5$) it is half the absolute error. For q90 it charges 0.9 per point when the truth lands **above** the forecast and only 0.1 per point when it lands below — so the best q90 is the value the truth exceeds 10% of the time. Averaging the five pinball losses rewards a forecast whose whole range is in the right place.

Two scalings make origins and horizons comparable: each row's loss is divided by $s_T\sqrt{h}$, where $s_T$ is the standard deviation of the 252 daily returns before the origin (known when you forecast). A score of 0.25 then means "a quarter of a typical move." The **benchmark** is the random walk with one-year volatility: median zero, quantiles $z_\tau\, s_T \sqrt{h}$.

The cell below builds the answer key and the benchmark for one origin — the last Thursday whose 20-day future is known before the cutoff — and scores two forecasts by hand: the benchmark and an **overconfident** version that is identical but half as wide.

In [ ]:
dates = close.index


def solution_for(origins):
    """Answer-key rows (id, y, h, scale, origin) for origins whose 20-day future is known."""
    rows = []
    for o in origins:
        i = dates.get_loc(o)
        s = ret.loc[:o].iloc[-252:].std(ddof=1)
        for h in HORIZONS:
            rows.append({"id": make_id(o, h), "y": 100 * np.log(close.iloc[i + h] / close.iloc[i]),
                         "h": h, "scale": s, "origin": f"{o:%Y-%m-%d}"})
    return pd.DataFrame(rows)


def benchmark_quantiles(past_ret, h):
    """The competition benchmark: random walk, one-year volatility, normal quantiles."""
    return Z * past_ret[-252:].std(ddof=1) * np.sqrt(h)


thursdays = [d for d in dates[:-max(HORIZONS)] if d.weekday() == 3]
demo_origin = thursdays[-1]
demo_sol = solution_for([demo_origin])
past = ret.loc[:demo_origin].to_numpy()
bench = pd.DataFrame([[make_id(demo_origin, h), *benchmark_quantiles(past, h)] for h in HORIZONS],
                     columns=["id"] + QCOLS)
overconfident = bench.copy()
overconfident[QCOLS] = bench[QCOLS] * 0.5

print(f"Origin: close of {demo_origin:%A %Y-%m-%d}")
print(demo_sol[["id", "y", "h", "scale"]].round(3).to_string(index=False))
print("\nBenchmark quantiles (percent):")
print(bench.round(3).to_string(index=False))
print(f"\nScore, benchmark              : {score(demo_sol, bench):.4f}")
print(f"Score, half-width overconfident: {score(demo_sol, overconfident):.4f}")

**Reading the output:** The origin is the close of **Thursday 1 October 2026**; the index then rose **0.73%** by the next close and ended **+0.70%** after 20 trading days, with a dip to about zero at 15 days. The benchmark's quantiles widen with $\sqrt{h}$: the 80% range (q10 to q90) is about ±1.05% for one day and ±4.69% for 20 days. Now the surprise: the **overconfident** forecast — the same quantiles at half the width — scored **0.1547**, *better* than the benchmark's **0.1724**. October 2026 was a calm month, every outcome landed near the middle, and the narrow range was rewarded. On one origin, overconfidence can win. That is the single-month lottery in miniature, and it is why the backtest below uses 104 origins: over many weeks, the turbulent ones arrive and an overconfident range pays for them in the tails.

> **A question that often comes up here:** *"Why not just score the median and ignore the rest?"* Because then a team that reads volatility well and a team that guesses blindly would tie whenever their medians match — and nearly every reasonable median is close to zero. A simulation of the competition design showed exactly that: point-error metrics cannot tell a volatility-aware forecast from a naive one at all, while the pinball loss over five quantiles can. With one live month the ranking is still close to a coin toss between two good methods, which is why your grade weights your walk-forward evidence far more than your rank.

---

## 4. Forecasters Are Functions of the Past Only

Every forecaster in this notebook has the same signature: it receives the **daily returns up to the origin close** (a NumPy array, in percent) and a horizon $h$, and returns five quantiles. It never receives a date or the full series, so it cannot peek at the future even by accident. The four candidates, declared before any backtest:

1. **Benchmark (incumbent)** — the competition's random walk with one-year volatility.
2. **EWMA volatility** — same median (zero), but the volatility is an exponentially weighted average of squared returns ($\lambda = 0.94$), which reacts faster to calm and turbulent stretches (nb16, Exercise 2).
3. **Drift + EWMA** — the EWMA band shifted by five years of average daily return times $h$: a bet that the market keeps rising at its usual pace.
4. **Empirical quantiles** — no normal curve at all: the 10th, 25th, 50th, 75th and 90th percentiles of every overlapping $h$-day return in the last three years. Fat tails and skew come for free; so does whatever drift those three years had.

In [ ]:
def ewma_vol(past_ret, lam=0.94, window=252):
    x = past_ret[-window:]
    w = lam ** np.arange(len(x))[::-1]          # the most recent day gets weight 1
    return np.sqrt(np.sum(w * x ** 2) / w.sum())


def fc_benchmark(past_ret, h):
    return benchmark_quantiles(past_ret, h)


def fc_ewma(past_ret, h):
    return Z * ewma_vol(past_ret) * np.sqrt(h)


def fc_drift_ewma(past_ret, h):
    return past_ret[-1260:].mean() * h + Z * ewma_vol(past_ret) * np.sqrt(h)


def fc_empirical(past_ret, h, window=756):
    x = past_ret[-window:]
    h_day = np.convolve(x, np.ones(h), mode="valid")      # every overlapping h-day sum
    return np.quantile(h_day, TAUS)


models = {"Benchmark (incumbent)": fc_benchmark, "EWMA volatility": fc_ewma,
          "Drift + EWMA": fc_drift_ewma, "Empirical quantiles": fc_empirical}

example = pd.DataFrame({name: f(past, 20) for name, f in models.items()}, index=QCOLS).T
print(f"20-day quantile forecasts (percent) at the origin {demo_origin:%Y-%m-%d}:")
print(example.round(2).to_string())

**Reading the output:** At the 1 October 2026 origin the four candidates disagree mostly about **width and center**, not shape. The benchmark's 20-day 80% range is **−4.68% to +4.68%**; EWMA volatility, reacting to the calm recent weeks, narrows it to **±3.62%**; Drift + EWMA shifts that narrower band up by about **0.86%** (five years of average drift times 20 days); and the empirical quantiles are both **shifted up** (median **+1.96%**) and **skewed** (−2.98% to +5.17%), because the last three years were mostly a rising market. Each forecaster encodes a different belief about the next month. The backtest asks which belief held up, origin after origin.

---

## 5. The Weekly Walk-Forward Backtest

The practice round and the live round both use **Thursday** origins, so the backtest does too: every Thursday close in the last two years whose 20-day future is known, **104 origins**. At each origin every model receives the same past and forecasts all five horizons; the same `score` function the leaderboard uses scores each origin. Because every model faces identical origins, the comparison with the benchmark can be **paired** origin by origin, exactly as in nb16.

In [ ]:
origins = thursdays[-104:]
solution = solution_for(origins)

per_origin = {}
for name, f in models.items():
    rows = []
    for o in origins:
        past_o = ret.loc[:o].to_numpy()                 # only returns up to the origin close
        rows += [[make_id(o, h), *f(past_o, h)] for h in HORIZONS]
    sub = pd.DataFrame(rows, columns=["id"] + QCOLS)
    per_origin[name] = [score(solution[solution.origin == f"{o:%Y-%m-%d}"],
                              sub[sub.id.str.startswith(f"{o:%Y-%m-%d}")]) for o in origins]
per_origin = pd.DataFrame(per_origin, index=origins)

bench_mean = per_origin["Benchmark (incumbent)"].mean()
summary = pd.DataFrame({"mean score": per_origin.mean(),
                        "skill vs benchmark": 1 - per_origin.mean() / bench_mean})
print(f"{len(origins)} Thursday origins: {origins[0]:%Y-%m-%d} -> {origins[-1]:%Y-%m-%d}")
print(summary.round(4).to_string())

gains = per_origin.drop(columns="Benchmark (incumbent)").rsub(per_origin["Benchmark (incumbent)"], axis=0)
print("\nPaired per-origin gains over the benchmark (score units; positive = better than the benchmark):")
print(pd.concat([describe_paired(gains[c], c) for c in gains.columns], axis=1).to_string())

fig, ax = plt.subplots(figsize=(12, 4.5))
for c in gains.columns:
    ax.plot(gains.index, gains[c].cumsum(), label=c)
ax.axhline(0, color="black", lw=1)
ax.set_ylabel("Running total of gains\nover the benchmark")
ax.set_title("Paired per-origin gains over the random-walk benchmark, accumulated week by week")
ax.legend()
plt.tight_layout()
plt.show()

**Reading the output:** Over **104 Thursday origins (29 August 2024 – 1 October 2026)** the benchmark's mean score is **0.258**. All three challengers do a little better on average — EWMA volatility **0.250** (skill 2.8%), Drift + EWMA **0.249** (3.5%), empirical quantiles **0.251** (2.5%) — and each wins **61–65** of the 104 origins. But read the spread before the mean. **EWMA volatility's gains have the smallest SD (0.029) and the mildest worst week (−0.059)**: it changes only the width of the band, so it rarely does much worse than the benchmark. **Drift + EWMA** has the highest mean gain (+0.009) but its gains are 45% more spread out (SD 0.042) and its worst week is twice as bad (−0.121): part of its edge is the bet that the market keeps rising, which paid over these two years. The **empirical quantiles** carry that same bet through their three-year window and have the worst single week (−0.135). The running totals show the shape: EWMA's line climbs steadily, the two drift-leaning lines climb faster and fall back in the sell-offs.

> **A question that often comes up here:** *"104 origins sounds like a lot. Why still no interval or verdict?"* Two reasons. The windows overlap — a 20-day forecast made this Thursday shares 15 trading days with the one made next Thursday — so 104 origins carry far less independent information than 104 separate months would. And the gains depend on the market's regime: a model that leans on drift wins in a rising market and loses in a falling one, and two years contain only a couple of regimes. Describing every origin, the mean, the spread and the running total shows the stakeholder that pattern directly. If you want a formal test, the Diebold–Mariano test (nb16) is built for exactly this kind of overlapping comparison; mention it in your memo as a "going further" check, not as the deciding rule.

---

## 6. Calibration Before Sharpness

A low score can come from a lucky center. Before choosing, check that each model's bands are **calibrated**: across the 104 origins, the truth should fall inside the q25–q75 band about 50% of the time and inside the q10–q90 band about 80% of the time, at every horizon.

In [ ]:
cov_rows = []
for name, f in models.items():
    for h in HORIZONS:
        in50, in80 = [], []
        for o in origins:
            q = f(ret.loc[:o].to_numpy(), h)
            y = solution.loc[solution.id == make_id(o, h), "y"].iloc[0]
            in50.append(q[1] <= y <= q[3])
            in80.append(q[0] <= y <= q[4])
        cov_rows.append({"model": name, "h": h, "inside q25-q75 (target 0.50)": np.mean(in50),
                         "inside q10-q90 (target 0.80)": np.mean(in80)})
coverage = pd.DataFrame(cov_rows)
print(coverage.pivot(index="model", columns="h", values="inside q10-q90 (target 0.80)").round(3)
      .to_string())
print()
print(coverage.pivot(index="model", columns="h", values="inside q25-q75 (target 0.50)").round(3)
      .to_string())

**Reading the output:** The first table is the share of origins whose realized return fell inside the **80% band** (q10–q90), the second inside the **50% band** (q25–q75), one column per horizon. All four models are close to calibrated, and none is overconfident at the 80% level beyond the 1-day horizon: the benchmark contains **79–87%** of outcomes in its 80% band, EWMA **81–88%**, Drift + EWMA **81–89%**, empirical **79–86%** — slightly conservative at the longer horizons, which over two mostly rising years is what a band centered at zero should show. At the 50% level, **EWMA's band is the closest to target at short horizons** (0.538, 0.510 and 0.548 at 1, 5 and 10 days, against the benchmark's 0.606, 0.558 and 0.606), while the empirical band is slightly too narrow at 1–5 days (0.481, 0.490). With 104 overlapping origins, differences of a few points are within noise; nothing here disqualifies a model.

**The treasury officer's decision.** The declared default is to keep the benchmark unless the described evidence makes the case for a challenger. For the **width** of the band it does: EWMA volatility scores better than the benchmark on average, with the steadiest gains, the mildest worst week, and calibration at least as close to target — and it rests on the one regularity nb16 showed is real, volatility clustering. For the **center** it does not: the extra gain of Drift + EWMA and of the empirical quantiles is a bet on the market's direction, which nb16 found unreadable and which these two rising years happened to reward. The treasury officer's champion statement:

> **Champion.** We recommend the **EWMA-volatility random walk** (median zero, $\lambda = 0.94$) for the pension fund's weekly cash-buffer range. **Evidence.** Against the random-walk benchmark on the same 104 Thursday origins, its mean score is 0.250 against 0.258 (skill 2.8%), with per-origin gains of +0.007 on average (SD 0.029), winning 63 of 104 weeks, worst week −0.059; we describe these differences rather than certify them, because the weekly windows overlap. **Checks.** Its 80% band contained 81–88% of outcomes and its 50% band 51–63% across the five horizons. **What would change our mind.** Drift + EWMA had a slightly higher mean gain but twice the worst-week loss; it is not an automatic fallback, and we would revisit a drift term only if it helped through a falling market as well as a rising one.

---

## 📝 PAUSE-AND-DO Exercise 1 — Fat tails: does a Student's t band beat the normal one? (10 minutes)

**Task:** Daily stock returns have fatter tails than the normal curve. Build a fifth forecaster, `fc_student_t(past_ret, h)`, that keeps the EWMA volatility but replaces the normal quantiles with **Student's t quantiles with 4 degrees of freedom**, rescaled so the distribution still has the EWMA standard deviation: $q_\tau = t_{4,\tau}\sqrt{(4-2)/4}\;\sigma_{\text{EWMA}}\sqrt{h}$. Backtest it on the same 104 origins, then describe its paired per-origin gains over **the benchmark** and over **EWMA volatility**. In two sentences: does the fat-tailed band help?

---

> 💡 **Gemini Prompt:** *"I have functions `ewma_vol(past_ret)`, `score(solution, submission)`, `make_id(origin, h)`, `describe_paired(gains, label)`, lists `HORIZONS` and `QCOLS`, an array `TAUS`, a list of Thursday dates `origins`, a DataFrame `solution` with columns id, y, h, scale, origin, a Series `ret` of daily returns in percent indexed by date, and a DataFrame `per_origin` with one column of per-origin scores per model. Write `fc_student_t(past_ret, h)` returning `stats.t.ppf(TAUS, 4) * np.sqrt(2 / 4) * ewma_vol(past_ret) * np.sqrt(h)`. Then, for each origin `o`, forecast all five horizons from `ret.loc[:o].to_numpy()`, score that origin with `score` on the matching solution rows, and store the per-origin scores as `per_origin["Student's t (4 df)"]`. Finally print `describe_paired` of the gains over 'Benchmark (incumbent)' and over 'EWMA volatility' (gain = other model's score minus the Student's t score)."*
>
> **After running, verify:**
> - [ ] `per_origin` now has a fifth column with 104 scores
> - [ ] A positive gain means the Student's t model had the lower (better) score at that origin
> - [ ] The q10 and q90 of the Student's t model are slightly *narrower* than the normal ones (with 4 df, rescaled to the same SD, the mass moves into the far tails and the center)

In [ ]:
# YOUR SOLUTION CODE HERE
# Hints:
# def fc_student_t(past_ret, h):
#     return stats.t.ppf(TAUS, 4) * np.sqrt(2 / 4) * ewma_vol(past_ret) * np.sqrt(h)
# Reuse the loop of Section 5 for one model, then
# describe_paired(per_origin["Benchmark (incumbent)"] - per_origin["Student's t (4 df)"], ...)

### YOUR ANSWER HERE:

**Does the fat-tailed band help? (two sentences, with the numbers):**

---

## 7. From Backtest to Submission Files

Two files leave your notebook, and both come from the **same forecasting function** you backtested.

- **The Kaggle practice submission.** Download `practice_origins.csv` from the competition's Data tab (it lists the practice origins and horizons — no index values). For each practice origin, forecast from the returns up to that origin's close, exactly as in the backtest, and upload the file on Kaggle. You know these outcomes already; that is fine — the practice round is ungraded and exists to rehearse the pipeline and the format.
- **The live forecast.** Set `DATA_END = "2027-03-25"`, re-run the whole notebook, and write `NN_live_forecast.csv` (your two-digit group number): five rows, one per horizon, ids `2027-03-25_h01` … `2027-03-25_h20`. Upload it to Brightspace with your notebook by **Fri Mar 26, 11:59 PM ET**.

The cell below writes a live-style file for the demo cutoff with the benchmark model and checks it with `validate_submission`, the same checks Kaggle and the live scorer apply.

In [ ]:
def validate_submission(sub, expected_ids):
    """The competition's format checks: exact columns, the expected ids once each, finite non-decreasing quantiles."""
    problems = []
    if list(sub.columns) != ["id"] + QCOLS:
        problems.append(f"columns must be exactly {['id'] + QCOLS}")
    if sub["id"].duplicated().any():
        problems.append("duplicate ids")
    if set(sub["id"]) != set(expected_ids):
        problems.append("ids differ from the expected rows")
    q = sub[QCOLS].to_numpy(dtype=float)
    if not np.isfinite(q).all():
        problems.append("blank, NaN or infinite quantiles")
    elif (np.diff(q, axis=1) < 0).any():
        problems.append("quantiles cross (need q10 <= q25 <= q50 <= q75 <= q90)")
    return problems


def forecast_file(forecaster, origin, past_ret):
    return pd.DataFrame([[make_id(origin, h), *forecaster(past_ret, h)] for h in HORIZONS],
                        columns=["id"] + QCOLS).round(4)


live_origin = close.index[-1]            # the last close at or before DATA_END
live = forecast_file(fc_benchmark, live_origin, ret.to_numpy())
expected = [make_id(live_origin, h) for h in HORIZONS]
print(f"Forecast origin: close of {live_origin:%A %Y-%m-%d} (= DATA_END's last trading day)")
print(live.to_string(index=False))
print("\nValidation problems:", validate_submission(live, expected) or "none - the file is valid")
live.to_csv("00_live_forecast_DEMO.csv", index=False)
print("Saved 00_live_forecast_DEMO.csv")

**Reading the output:** The live-style file has the required six columns and five rows, ids `2026-10-30_h01` … `2026-10-30_h20`, and passes every check. With the benchmark model the median is zero at every horizon and the quantiles grow with $\sqrt{h}$: the 80% range is ±1.04% for one day and ±4.63% for 20 days, set by the last year's volatility (0.808% a day). The demo file is named `00_…_DEMO` so it can never be mistaken for a group's real submission. For the live forecast your origin will be the close of **Thursday, March 25, 2027**, and the ids will be `2027-03-25_h01` … `2027-03-25_h20`.

---

## 📝 PAUSE-AND-DO Exercise 2 — Write and prove your group's submission file (10 minutes)

**Task:** Choose the forecaster your group would submit today (any model from Sections 4–6 or Exercise 1, or your own). (1) Write a live-style file for the demo cutoff named with your group number, `NN_live_forecast.csv`, and validate it. (2) Prove it is **reproducible**: generate it a second time from scratch and confirm the two DataFrames are identical. (3) Prove there is **no look-ahead**: confirm that its origin is on or before `DATA_END`. Then write the one-sentence champion statement your memo would open with.

---

> 💡 **Gemini Prompt:** *"Using my functions `forecast_file(forecaster, origin, past_ret)` and `validate_submission(sub, expected_ids)`, my forecaster `fc_ewma`, the Series `ret` of daily percent returns, `close` (the closing levels), `make_id`, `HORIZONS` and the string `DATA_END`: set `GROUP = "07"`, build `sub = forecast_file(fc_ewma, close.index[-1], ret.to_numpy())`, validate it against the five expected ids, save it as `f'{GROUP}_live_forecast.csv'` without the index, rebuild it a second time and check `sub.equals(sub_again)`, and assert that `close.index[-1] <= pd.Timestamp(DATA_END)`. Print the file, the validation result, the equality check and the origin date."*
>
> **After running, verify:**
> - [ ] The file has exactly six columns (`id`, `q10` … `q90`) and five rows
> - [ ] Validation returns no problems
> - [ ] The rebuild is identical (`True`) and the origin is on or before the cutoff

In [ ]:
# YOUR SOLUTION CODE HERE
# Hints:
# GROUP = "07"                                   # your two-digit group number
# sub = forecast_file(fc_ewma, close.index[-1], ret.to_numpy())
# validate_submission(sub, [make_id(close.index[-1], h) for h in HORIZONS])
# sub.to_csv(f"{GROUP}_live_forecast.csv", index=False)

### YOUR ANSWER HERE:

**Our model and why (one-sentence champion statement):**

---

## 8. What Your Group Submits, and How It Is Graded

The competition rewards **method and honesty** far more than **rank**, because one live month is mostly luck (Section 3). Your group submits, by **Fri Mar 26, 2027, 11:59 PM ET**:

1. **`NN_live_forecast.csv`** — the five live rows, validated.
2. **`NN_competition_code.ipynb`** — this workflow adapted to your model: `DATA_END = "2027-03-25"`, the weekly walk-forward backtest on at least 52 Thursday origins with the benchmark and at least two challengers, the paired per-origin differences described, the calibration table, and the cell that writes your live file. A fresh **Runtime → Run all** must regenerate the same file.
3. **`NN_forecast_memo.pdf`** — one to two pages for the treasury officer: your champion statement against the random walk, the calibration evidence, and the range of scores your backtest leads you to expect for the live month — written **before** the month happens.

Your group also makes at least one valid submission to the **Kaggle practice round** under the team name `Group NN`. The weights, the rubric and the penalties are in the competition instructions on Brightspace.

> **A question that often comes up here:** *"Can we wait until Monday, March 29 to submit, so we know one more day?"* No — and the reason is the whole point of forecasting. A forecast is only a forecast if it is fixed before the outcome exists. The deadline is the evening of Good Friday, when the market is closed and the last close you may use (Thursday, March 25) is already published; a forecast that knew anything later would be measuring hindsight, not skill.

---

## 9. Wrap-Up: Key Takeaways

### What We Learned Today:

1. **A forecast has an information cutoff.** `DATA_END` and forecasters that only see the past make look-ahead impossible by construction.
2. **Quantiles describe the range; the pinball loss scores it.** Scaling by $s_T\sqrt{h}$ puts every origin and horizon on the scale of a typical move.
3. **Backtest weekly, on identical origins, against the benchmark.** Describe the paired per-origin gains, the spread and the running total; do not let one lucky stretch decide.
4. **Calibration first, sharpness second.** A band that is too narrow loses more in the tails than it gains in the middle.
5. **Ship a file you can prove.** Validated, reproducible, and generated by the same function you backtested.

### Remember:

> **"The market's direction is mostly luck; the honesty of your range is skill."**

**Next: nb19 — Deep Learning.** The last technical notebook brings neural networks into the same discipline you have used all term: an incumbent, identical folds, and a challenger that has to earn its place. Meanwhile, the competition runs in the background: team setup on Kaggle by **Sun Mar 7**, the practice round until **Thu Mar 25**, and your live forecast, code and memo on **Fri Mar 26**.

> **A question that often comes up here:** *"What if our backtest says we cannot beat the benchmark?"* Then say so, and submit the forecast your evidence supports — possibly the benchmark itself, or the benchmark with a better volatility estimate. A memo that states plainly "our challengers did not improve on the random walk over 104 weeks, so we submit a calibrated random-walk range" is exactly the honest comparison the treasury officer needs and the rubric rewards. Claiming an edge your backtest does not show is the one thing that costs you.

---

## 10. Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete both exercises**: Fill in both PAUSE-AND-DO exercise cells with your code and your answers
2. **Run All Cells**: Execute `Runtime → Run all` to ensure everything works
3. **Save a Copy**: `File → Save a copy in Drive or Download the .ipynb extension`
4. **Submit**: Upload your `.ipynb` file in the participation assignment you find in the course Brightspace page.

### Before Submitting, Check:

- [ ] All cells execute without errors
- [ ] All outputs are visible
- [ ] Both exercise responses are complete
- [ ] You can explain every line of code you wrote

---

## Bibliography

- Gneiting, T., & Raftery, A. E. (2007). Strictly proper scoring rules, prediction, and estimation. *Journal of the American Statistical Association*, 102(477), 359–378.
- Bracher, J., Ray, E. L., Gneiting, T., & Reich, N. G. (2021). Evaluating epidemic forecasts in an interval format. *PLOS Computational Biology*, 17(2), e1008618.
- Diebold, F. X., & Mariano, R. S. (1995). Comparing predictive accuracy. *Journal of Business & Economic Statistics*, 13(3), 253–263.
- Hyndman, R. J., & Athanasopoulos, G. (2021). *Forecasting: Principles and Practice* (3rd ed.). OTexts. https://otexts.com/fpp3/ — §5.9 (evaluating distributional forecast accuracy: quantile scores).
- Makridakis, S., Spiliotis, E., & Assimakopoulos, V. (2022). The M5 competition: Background, organization, and implementation. *International Journal of Forecasting*, 38(4), 1325–1336.
- J.P. Morgan/Reuters (1996). *RiskMetrics — Technical Document* (4th ed.).

---

<center>

Thank you!

</center>